In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import torchvision.transforms as transforms
import pandas as pd
import numpy as np
from PIL import Image
import os
from tqdm.notebook import tqdm as tqdm_notebook

# 2. Define the image preprocessing pipeline
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 3. Load the CSV file
df_predictions = pd.read_csv('/content/drive/MyDrive/XAI_Project/results/combined_predictions/all_models_combined_cat_predictions_without_nan.csv')
print(f"Loaded predictions DataFrame with {len(df_predictions)} rows.")

# 4. Define the base path for input images
IMAGE_BASE_DIR = '/content/drive/MyDrive/XAI_Project/cats_images'
print(f"Image base directory set to: {IMAGE_BASE_DIR}")

# 5. Define the base path for output saliency maps
OUTPUT_BASE_DIR = '/content/drive/MyDrive/XAI_Project/saliency_maps_last'
print(f"Output base directory set to: {OUTPUT_BASE_DIR}")

In [ ]:
import os
import torchvision.models as models
import json

# 1. Define MODELS_LIST
#MODELS_LIST = ['mobilenet_v3_large', 'efficientnet_b0', 'efficientnet_b3', 'squeezenet', 'resnet18']
MODELS_LIST = ['efficientnet_b3', 'squeezenet', 'resnet18']


# 2. Create a dictionary to map model names to their corresponding loading functions
MODEL_LOADERS = {
    #'mobilenet_v3_large': lambda: models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V1),
    #'efficientnet_b0': lambda: models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1),
    'efficientnet_b3': lambda: models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1),
    'squeezenet': lambda: models.squeezenet1_0(weights=models.SqueezeNet1_0_Weights.IMAGENET1K_V1),
    'resnet18': lambda: models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
}

print("Defined MODELS_LIST and MODEL_LOADERS.")

# Load ImageNet class labels for get_imagenet_class_id function
# A common way to get ImageNet class labels is from a JSON file.
# If not available locally, we will fetch it or use a simplified mapping.
# For demonstration, we'll use a direct mapping or a placeholder.
# In a real scenario, you'd download 'imagenet_class_index.json' if needed.

# Placeholder for ImageNet class index (in a real scenario, load from a file)
# This is a simplified version; a full ImageNet class index has 1000 entries.
# You might need to download 'imagenet_class_index.json' and load it.
# For now, let's create a minimal one based on common cat breeds.
# A more robust solution would involve downloading the actual class index.

# Try to download imagenet_class_index.json if not present
# This part assumes a common location for ImageNet class indices.
# If this fails, a manual mapping would be needed.

# --- BEGIN: Fetch ImageNet class index if not available locally ---
IMAGENET_CLASS_INDEX_PATH = '/tmp/imagenet_classes.txt' # Changed extension to .txt
if not os.path.exists(IMAGENET_CLASS_INDEX_PATH):
    try:
        import urllib.request
        url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
        urllib.request.urlretrieve(url, IMAGENET_CLASS_INDEX_PATH)
        print("Downloaded ImageNet class index.")
    except Exception as e:
        print(f"Could not download ImageNet class index: {e}")
        print("Using a simplified placeholder for cat breeds. This might lead to incorrect class IDs if the model predicts other classes.")
        # Fallback to a minimal hardcoded dictionary if download fails
        imagenet_class_index = {
            "281": ["n02123597", "Siamese_cat"],
            "282": ["n02123975", "tabby"],
            "283": ["n02124075", "tiger_cat"],
            "285": ["n02125311", "domestic_cat"],
            "286": ["n02127052", "lynx"],
            "287": ["n02128757", "snow_leopard"],
            "288": ["n02129165", "lion"],
            "289": ["n02129604", "tiger"],
            "290": ["n02130308", "cheetah"],
            "291": ["n02131653", "Pallas'_cat"],
            "292": ["n02132136", "jaguar"],
            "284": ["n02124075", "Egyptian_cat"], # Often mapped to tiger_cat or similar
            "800": ["n04252077", "Persian_cat"] # Persian cat is not directly in original ImageNet, often a generic cat class is used. Using a common index for demonstration.
        }
else:
    print("ImageNet class index already exists locally.")

imagenet_class_index = {}
if os.path.exists(IMAGENET_CLASS_INDEX_PATH):
    with open(IMAGENET_CLASS_INDEX_PATH) as f:
        # Read the file line by line and create a dictionary with integer indices
        idx2label = {str(i): [None, line.strip()] for i, line in enumerate(f)}
    # Reformat to be label_name -> index
    # The ImageNet classes.txt file has one label per line, so we need to adjust how we parse it.
    # We'll create a mapping from label name to its corresponding integer index.
    imagenet_class_index = {class_info[1]: int(idx) for idx, class_info in idx2label.items()}
else:
    # Use the hardcoded minimal dictionary if download failed and file doesn't exist
    for idx_str, class_info in imagenet_class_index.items(): # using the fallback dict from above
        imagenet_class_index[class_info[1].replace('_', ' ')] = int(idx_str)


# 3. Define a helper function get_imagenet_class_id(class_name)
def get_imagenet_class_id(class_name):
    # ImageNet labels might use underscores instead of spaces, so handle that
    formatted_class_name = class_name.replace(' ', '_').lower()
    for label, idx in imagenet_class_index.items():
        if formatted_class_name in label.lower() or label.lower() in formatted_class_name:
            return idx
    # Fallback for common cat breeds if exact match not found
    if 'cat' in formatted_class_name:
        if 'tabby' in formatted_class_name: return imagenet_class_index.get('tabby', 282)
        if 'egyptian' in formatted_class_name: return imagenet_class_index.get('Egyptian_cat', imagenet_class_index.get('tabby', 282)) # Often maps to tabby or a specific cat type
        if 'persian' in formatted_class_name: return imagenet_class_index.get('Persian_cat', imagenet_class_index.get('Persian cat', 800)) # Placeholder
        return imagenet_class_index.get('domestic_cat', 285) # Generic cat class
    return None # If no match found

print("Defined get_imagenet_class_id function.")

# 4. Define a helper function save_saliency_map
def save_saliency_map(saliency_map, output_dir, filename_prefix, image_name):
    # a. Normalize the saliency map to the range 0-255
    saliency_map_np = saliency_map.cpu().numpy()
    saliency_map_np = (saliency_map_np - saliency_map_np.min()) / (saliency_map_np.max() - saliency_map_np.min() + 1e-8) * 255
    saliency_map_np = saliency_map_np.astype(np.uint8)

    # b. Convert to PIL Image
    saliency_img = Image.fromarray(saliency_map_np, mode='L') # Assuming H x W

    # c. Create the output directory if it doesn't already exist
    os.makedirs(output_dir, exist_ok=True)

    # d. Save as JPG
    jpg_path = os.path.join(output_dir, f'{os.path.splitext(image_name)[0]}_{filename_prefix}_saliency.jpg')
    saliency_img.save(jpg_path)

    # e. Save as NPY
    npy_path = os.path.join(output_dir, f'{os.path.splitext(image_name)[0]}_{filename_prefix}_saliency.npy')
    np.save(npy_path, saliency_map.cpu().numpy())

    # print(f"Saved saliency map to {jpg_path} and {npy_path}")

print("Defined save_saliency_map function.")

In [ ]:
import torch
import torch.nn.functional as F
from scipy.ndimage import median_filter

def vanilla_saliency(model, input_img, target_class, median_size=5):
    model.eval()
    input_img = input_img.clone().detach().requires_grad_(True)

    # Forward pass
    output = model(input_img)
    # Select the score for the target class
    score = output[0, target_class]

    model.zero_grad()
    score.backward() # Compute gradients

    saliency = input_img.grad.abs()
    saliency, _ = saliency.max(dim=1)  # Take max across color channels (1 x H x W)

    # Apply median filter and normalize, similar to integrated_gradients and smoothgrad
    saliency_np = saliency.squeeze(0).cpu().numpy()
    saliency_med = median_filter(saliency_np, size=median_size)

    s_min, s_max = saliency_med.min(), saliency_med.max()
    if s_max > s_min:
        saliency_norm = (saliency_med - s_min) / (s_max - s_min)
    else:
        saliency_norm = saliency_med

    # Return to tensor H x W
    saliency_norm = torch.tensor(saliency_norm, device=input_img.device)

    return saliency_norm

In [ ]:
import torch
import numpy as np
from scipy.ndimage import median_filter

def integrated_gradients(model, input_img, target_class, baseline=None, steps=50, median_size=5):
    model.eval()

    if baseline is None:
        baseline = torch.zeros_like(input_img)

    # 1. Tworzenie interpolowanych wejść
    scaled_inputs = [
        baseline + (float(i) / steps) * (input_img - baseline)
        for i in range(1, steps + 1)
    ]

    grads = []
    for x in scaled_inputs:
        x = x.clone().detach().requires_grad_(True)
        output = model(x)
        score = output[0, target_class]

        model.zero_grad()
        score.backward()
        grads.append(x.grad.detach())

    # 2. Średni gradient
    avg_grads = torch.mean(torch.stack(grads), dim=0)

    # 3. Integrated gradients
    ig = (input_img - baseline) * avg_grads
    ig, _ = ig.abs().max(dim=1)  # 1 x H x W

    # 4. Filtr medianowy
    ig_np = ig.squeeze(0).cpu().numpy()
    ig_med = median_filter(ig_np, size=median_size)

    # 5. Normalizacja do 0–1
    ig_min, ig_max = ig_med.min(), ig_med.max()
    if ig_max > ig_min:
        ig_norm = (ig_med - ig_min) / (ig_max - ig_min)
    else:
        ig_norm = ig_med  # obraz stały — zostawiamy jak jest

    # powrót do tensora H x W
    ig_norm = torch.tensor(ig_norm)

    return ig_norm

In [ ]:
import torch
import numpy as np
from scipy.ndimage import median_filter

def smoothgrad(
    model,
    input_img,
    target_class,
    n_samples=50,
    noise_level=0.1,
    median_size=5,
):
    model.eval()

    _, _, H, W = input_img.shape
    saliency_sum = torch.zeros((1, H, W), device=input_img.device)

    for _ in range(n_samples):
        noise = noise_level * torch.randn_like(input_img)
        noisy_img = (input_img + noise).clone().detach().requires_grad_(True)

        output = model(noisy_img)
        score = output[0, target_class]

        model.zero_grad()
        score.backward()

        grad = noisy_img.grad.abs()
        grad, _ = grad.max(dim=1)  # po kanałach
        saliency_sum += grad.detach()

    # 1. Średnia z próbek
    saliency = saliency_sum / n_samples  # tensor 1 x H x W

    # 2. Filtr medianowy
    sal_np = saliency.squeeze(0).cpu().numpy()
    sal_med = median_filter(sal_np, size=median_size)

    # 3. Normalizacja do 0–1
    s_min, s_max = sal_med.min(), sal_med.max()
    if s_max > s_min:
        sal_norm = (sal_med - s_min) / (s_max - s_min)
    else:
        sal_norm = sal_med

    # 4. Powrót do tensora H x W
    sal_norm = torch.tensor(sal_norm, device=input_img.device)

    return sal_norm

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def find_file_in_subdirs(base_dir, filename_to_find):
    for root, _, files in os.walk(base_dir):
        if filename_to_find in files:
            return os.path.join(root, filename_to_find)
    return None


# 5. Iterate through each model
for model_name in MODELS_LIST:
    print(f"\nProcessing model: {model_name}")

    # 6a. Load the pre-trained model and move to device
    model = MODEL_LOADERS[model_name]().to(device)
    model.eval()

    # Iterate through each row in df_predictions
    for idx, row in tqdm_notebook(
        df_predictions.iterrows(),
        total=len(df_predictions),
        desc=f"Generating saliency for {model_name}"
    ):
        filename = row["filename"]
        predicted_label = row[f"{model_name}_top_1_class"]

        # Try direct path
        current_image_path = os.path.join(IMAGE_BASE_DIR, filename)

        if os.path.exists(current_image_path):
            full_image_path = current_image_path
        else:
            # Try searching subdirectories
            found_path = find_file_in_subdirs(IMAGE_BASE_DIR, filename)
            if found_path is None:
                print(f"Warning: Image not found: {filename}. Skipping.")
                continue
            full_image_path = found_path

        # Load image
        img = Image.open(full_image_path).convert("RGB")

        # Preprocess
        input_tensor = preprocess(img).unsqueeze(0).to(device)

        # Get ImageNet class ID
        target_class_id = get_imagenet_class_id(predicted_label)
        if target_class_id is None:
            print(f"Warning: No ImageNet ID for '{predicted_label}' ({filename}). Skipping.")
            continue

        # 6h. Generate saliency maps
        vanilla_saliency_map = vanilla_saliency(
            model, input_tensor.clone(), target_class_id
        )
        print(f"Vanilla Saliency Map Shape: {vanilla_saliency_map.shape}")

        integrated_saliency_map = integrated_gradients(
            model, input_tensor.clone(), target_class_id
        )
        print(f"Integrated Gradients Saliency Map Shape: {integrated_saliency_map.shape}")

        smoothgrad_saliency_map = smoothgrad(
            model, input_tensor.clone(), target_class_id
        )
        print(f"SmoothGrad Saliency Map Shape: {smoothgrad_saliency_map.shape}")

        # 6i. Save saliency maps
        vanilla_output_dir = os.path.join(OUTPUT_BASE_DIR, model_name, "vanilla_saliency")
        save_saliency_map(vanilla_saliency_map, vanilla_output_dir, "vanilla", filename)

        integrated_output_dir = os.path.join(OUTPUT_BASE_DIR, model_name, "integrated_gradients")
        save_saliency_map(integrated_saliency_map, integrated_output_dir, "integrated", filename)

        smoothgrad_output_dir = os.path.join(OUTPUT_BASE_DIR, model_name, "smoothgrad")
        save_saliency_map(smoothgrad_saliency_map, smoothgrad_output_dir, "smoothgrad", filename)

print("Saliency map generation complete.")

# Task
Add `import os` to cell `04bc690d`, then re-run cells `3a214001` and `04bc690d` to ensure `preprocess` with `CenterCrop(224)` and `save_saliency_map` are correctly loaded. Finally, confirm that all saliency map generation functions return maps of shape `(H, W)`, `save_saliency_map` processes them correctly, and images are cropped to `224x224`.

## Naprawa błędu importu 'os'

### Subtask:
Dodaj `import os` na początku komórki `04bc690d`, aby rozwiązać błąd `NameError: name 'os' is not defined`.


# Task
Re-run the preprocessing and saliency map saving functions to ensure they are up-to-date, add shape verification print statements in the saliency map generation loop, and then confirm that all generated saliency maps are `(H, W)` tensors which are saved as `224x224` JPG images, effectively addressing the issue of `(1, H, W)` tensors.

## Re-run preprocess and save_saliency_map definitions

### Subtask:
Re-run cells `3a214001` and `04bc690d` to ensure the latest definitions of `preprocess` (with `CenterCrop(224)`) and `save_saliency_map` (with the `import os` fix and correct handling of (H, W) input) are active. This is a crucial step to ensure the environment reflects the intended code changes.


**Reasoning**:
The subtask requires re-running cell `3a214001` to ensure the `preprocess` function is correctly defined. This is the first step in the provided instructions.



In [ ]:
import torch
import torchvision.transforms as transforms
import pandas as pd
import numpy as np
from PIL import Image
import os
from tqdm.notebook import tqdm as tqdm_notebook

# 2. Define the image preprocessing pipeline
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 3. Load the CSV file
df_predictions = pd.read_csv('/content/drive/MyDrive/XAI_Project/results/combined_predictions/all_models_combined_cat_predictions_without_nan.csv')
print(f"Loaded predictions DataFrame with {len(df_predictions)} rows.")

# 4. Define the base path for input images
IMAGE_BASE_DIR = '/content/drive/MyDrive/XAI_Project/cats_images'
print(f"Image base directory set to: {IMAGE_BASE_DIR}")

# 5. Define the base path for output saliency maps
OUTPUT_BASE_DIR = '/content/drive/MyDrive/XAI_Project/saliency_maps_last/Milek'
print(f"Output base directory set to: {OUTPUT_BASE_DIR}")

**Reasoning**:
The subtask requires re-running cell `04bc690d` to ensure the `save_saliency_map` function is defined with the correct `import os` and logic. This is the second step in the provided instructions.



In [ ]:
import os
import torchvision.models as models
import json

# 1. Define MODELS_LIST
#MODELS_LIST = ['mobilenet_v3_large', 'efficientnet_b0', 'efficientnet_b3', 'squeezenet', 'resnet18']
MODELS_LIST = ['efficientnet_b3', 'squeezenet', 'resnet18']


# 2. Create a dictionary to map model names to their corresponding loading functions
MODEL_LOADERS = {
    #'mobilenet_v3_large': lambda: models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V1),
    #'efficientnet_b0': lambda: models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1),
    'efficientnet_b3': lambda: models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1),
    'squeezenet': lambda: models.squeezenet1_0(weights=models.SqueezeNet1_0_Weights.IMAGENET1K_V1),
    'resnet18': lambda: models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
}

print("Defined MODELS_LIST and MODEL_LOADERS.")

# Load ImageNet class labels for get_imagenet_class_id function
# A common way to get ImageNet class labels is from a JSON file.
# If not available locally, we will fetch it or use a simplified mapping.
# For demonstration, we'll use a direct mapping or a placeholder.
# In a real scenario, you'd download 'imagenet_class_index.json' if needed.

# Placeholder for ImageNet class index (in a real scenario, load from a file)
# This is a simplified version; a full ImageNet class index has 1000 entries.
# You might need to download 'imagenet_class_index.json' and load it.
# For now, let's create a minimal one based on common cat breeds.
# A more robust solution would involve downloading the actual class index.

# Try to download imagenet_class_index.json if not present
# This part assumes a common location for ImageNet class indices.
# If this fails, a manual mapping would be needed.

# --- BEGIN: Fetch ImageNet class index if not available locally ---
IMAGENET_CLASS_INDEX_PATH = '/tmp/imagenet_classes.txt' # Changed extension to .txt
if not os.path.exists(IMAGENET_CLASS_INDEX_PATH):
    try:
        import urllib.request
        url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
        urllib.request.urlretrieve(url, IMAGENET_CLASS_INDEX_PATH)
        print("Downloaded ImageNet class index.")
    except Exception as e:
        print(f"Could not download ImageNet class index: {e}")
        print("Using a simplified placeholder for cat breeds. This might lead to incorrect class IDs if the model predicts other classes.")
        # Fallback to a minimal hardcoded dictionary if download fails
        imagenet_class_index = {
            "281": ["n02123597", "Siamese_cat"],
            "282": ["n02123975", "tabby"],
            "283": ["n02124075", "tiger_cat"],
            "285": ["n02125311", "domestic_cat"],
            "286": ["n02127052", "lynx"],
            "287": ["n02128757", "snow_leopard"],
            "288": ["n02129165", "lion"],
            "289": ["n02129604", "tiger"],
            "290": ["n02130308", "cheetah"],
            "291": ["n02131653", "Pallas'_cat"],
            "292": ["n02132136", "jaguar"],
            "284": ["n02124075", "Egyptian_cat"], # Often mapped to tiger_cat or similar
            "800": ["n04252077", "Persian_cat"] # Persian cat is not directly in original ImageNet, often a generic cat class is used. Using a common index for demonstration.
        }
else:
    print("ImageNet class index already exists locally.")

imagenet_class_index = {}
if os.path.exists(IMAGENET_CLASS_INDEX_PATH):
    with open(IMAGENET_CLASS_INDEX_PATH) as f:
        # Read the file line by line and create a dictionary with integer indices
        idx2label = {str(i): [None, line.strip()] for i, line in enumerate(f)}
    # Reformat to be label_name -> index
    # The ImageNet classes.txt file has one label per line, so we need to adjust how we parse it.
    # We'll create a mapping from label name to its corresponding integer index.
    imagenet_class_index = {class_info[1]: int(idx) for idx, class_info in idx2label.items()}
else:
    # Use the hardcoded minimal dictionary if download failed and file doesn't exist
    for idx_str, class_info in imagenet_class_index.items(): # using the fallback dict from above
        imagenet_class_index[class_info[1].replace('_', ' ')] = int(idx_str)


# 3. Define a helper function get_imagenet_class_id(class_name)
def get_imagenet_class_id(class_name):
    # ImageNet labels might use underscores instead of spaces, so handle that
    formatted_class_name = class_name.replace(' ', '_').lower()
    for label, idx in imagenet_class_index.items():
        if formatted_class_name in label.lower() or label.lower() in formatted_class_name:
            return idx
    # Fallback for common cat breeds if exact match not found
    if 'cat' in formatted_class_name:
        if 'tabby' in formatted_class_name: return imagenet_class_index.get('tabby', 282)
        if 'egyptian' in formatted_class_name: return imagenet_class_index.get('Egyptian_cat', imagenet_class_index.get('tabby', 282)) # Often maps to tabby or a specific cat type
        if 'persian' in formatted_class_name: return imagenet_class_index.get('Persian_cat', imagenet_class_index.get('Persian cat', 800)) # Placeholder
        return imagenet_class_index.get('domestic_cat', 285) # Generic cat class
    return None # If no match found

print("Defined get_imagenet_class_id function.")

# 4. Define a helper function save_saliency_map
def save_saliency_map(saliency_map, output_dir, filename_prefix, image_name):
    # a. Normalize the saliency map to the range 0-255
    saliency_map_np = saliency_map.cpu().numpy()
    saliency_map_np = (saliency_map_np - saliency_map_np.min()) / (saliency_map_np.max() - saliency_map_np.min() + 1e-8) * 255
    saliency_map_np = saliency_map_np.astype(np.uint8)

    # b. Convert to PIL Image
    saliency_img = Image.fromarray(saliency_map_np, mode='L') # Assuming H x W

    # c. Create the output directory if it doesn't already exist
    os.makedirs(output_dir, exist_ok=True)

    # d. Save as JPG
    jpg_path = os.path.join(output_dir, f'{os.path.splitext(image_name)[0]}_{filename_prefix}_saliency.jpg')
    saliency_img.save(jpg_path)

    # e. Save as NPY
    npy_path = os.path.join(output_dir, f'{os.path.splitext(image_name)[0]}_{filename_prefix}_saliency.npy')
    np.save(npy_path, saliency_map.cpu().numpy())

    # print(f"Saved saliency map to {jpg_path} and {npy_path}")

print("Defined save_saliency_map function.")

In [ ]:
import torch
import torch.nn.functional as F
from PIL import Image
import os
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Re-use already defined functions and variables from previous cells
# preprocess, MODEL_LOADERS, get_imagenet_class_id, save_saliency_map
# vanilla_saliency, integrated_gradients, smoothgrad, device, imagenet_class_index

# 1. Define the input image path and output directory
INPUT_IMAGE_PATH = '/content/1773665077339 (1).jpg'
OUTPUT_BASE_DIR_CUSTOM = '/content/drive/MyDrive/XAI_Project/saliency_maps_last/Milek'
MODEL_NAME_CUSTOM = 'resnet18' # Assuming 'efficientnet b03' was a typo for 'efficientnet_b3'

print(f"Processing image: {INPUT_IMAGE_PATH}")
print(f"Using model: {MODEL_NAME_CUSTOM}")
print(f"Saving results to: {OUTPUT_BASE_DIR_CUSTOM}")

# Ensure the output directory exists
os.makedirs(OUTPUT_BASE_DIR_CUSTOM, exist_ok=True)

# 2. Load the specified model
model = MODEL_LOADERS[MODEL_NAME_CUSTOM]().to(device)
model.eval()
print(f"Model '{MODEL_NAME_CUSTOM}' loaded successfully.")

# 3. Load and preprocess the input image
img = Image.open(INPUT_IMAGE_PATH).convert("RGB")
input_tensor = preprocess(img).unsqueeze(0).to(device)
print(f"Image '{INPUT_IMAGE_PATH}' loaded and preprocessed.")

# 4. Get the model's prediction for the image to determine the target_class_id
with torch.no_grad():
    output = model(input_tensor)
    probabilities = F.softmax(output, dim=1)
    top_prob, top_cat_id = torch.max(probabilities, 1)

    # Get the human-readable label from the ImageNet class index using top_cat_id
    # Create a reverse mapping from index to label name
    idx_to_label = {idx: label_name for label_name, idx in imagenet_class_index.items()}
    predicted_label = idx_to_label.get(top_cat_id.item(), 'unknown_class')

# Get ImageNet class ID for the predicted label
target_class_id = get_imagenet_class_id(predicted_label)

if target_class_id is None:
    print(f"Warning: Could not determine ImageNet ID for predicted label '{predicted_label}'. Trying a generic cat class.")
    # Fallback to a generic cat class if the predicted label isn't found
    target_class_id = get_imagenet_class_id('tabby cat') # 'tabby cat' often maps to ID 282
    if target_class_id is None:
        print("Error: Could not find a suitable target class ID. Cannot generate saliency maps.")
    else:
        print(f"Using fallback target class ID: {target_class_id} for 'tabby cat'.")
else:
    print(f"Predicted label: '{predicted_label}', ImageNet ID: {target_class_id}")


if target_class_id is not None:
    # 5. Generate saliency maps
    print("Generating Vanilla Saliency map...")
    vanilla_saliency_map = vanilla_saliency(
        model, input_tensor.clone(), target_class_id
    )
    print(f"Vanilla Saliency Map Shape: {vanilla_saliency_map.shape}")

    print("Generating Integrated Gradients map...")
    integrated_saliency_map = integrated_gradients(
        model, input_tensor.clone(), target_class_id
    )
    print(f"Integrated Gradients Saliency Map Shape: {integrated_saliency_map.shape}")

    print("Generating SmoothGrad map...")
    smoothgrad_saliency_map = smoothgrad(
        model, input_tensor.clone(), target_class_id
    )
    print(f"SmoothGrad Saliency Map Shape: {smoothgrad_saliency_map.shape}")

    # 6. Save the generated maps
    image_filename = os.path.basename(INPUT_IMAGE_PATH)

    # Save Vanilla Saliency
    vanilla_output_dir = os.path.join(OUTPUT_BASE_DIR_CUSTOM, MODEL_NAME_CUSTOM, "vanilla_saliency")
    save_saliency_map(vanilla_saliency_map, vanilla_output_dir, "vanilla", image_filename)
    print(f"Vanilla Saliency map saved to {vanilla_output_dir}")

    # Save Integrated Gradients
    integrated_output_dir = os.path.join(OUTPUT_BASE_DIR_CUSTOM, MODEL_NAME_CUSTOM, "integrated_gradients")
    save_saliency_map(integrated_saliency_map, integrated_output_dir, "integrated", image_filename)
    print(f"Integrated Gradients map saved to {integrated_output_dir}")

    # Save SmoothGrad
    smoothgrad_output_dir = os.path.join(OUTPUT_BASE_DIR_CUSTOM, MODEL_NAME_CUSTOM, "smoothgrad")
    save_saliency_map(smoothgrad_saliency_map, smoothgrad_output_dir, "smoothgrad", image_filename)
    print(f"SmoothGrad map saved to {smoothgrad_output_dir}")

    print("\nSaliency map generation for specified image complete.")
else:
    print("Saliency map generation skipped due to missing target class ID.")
